# Agentic AI — Local Live Demo (Phi-3 Mini)

Real production pipeline for the **Phi-3 Mini** adapter on **both frameworks** (Cypress, then Playwright): Sections 2–5 (Build → Measure → Syntax → BMAD loop) for each, with an explicit memory-management eviction between them (Section 4.5.2).

> ⚠️ **Run this notebook ALONE.** It loads real Phi-3 Mini adapters into one kernel. Before running it, **shut down any other model notebook's kernel** (Jupyter: *Kernel → Shut Down Kernel*) — otherwise both models stay resident at once and the kernel will crash (MPS does not release memory until the process exits). This is the notebook-level equivalent of the chunked-execution / process-restart strategy in Section 4.5.3.

## 0. Environment setup + memory-management helpers

In [1]:
import sys, os
REPO = "/Users/saif.afzal/Documents/Dissertation/agentic-test-gen"
sys.path.insert(0, REPO); os.chdir(REPO)

import gc, torch
from agentic_loop import generator
from agentic_loop.generator import generate as real_generate
from agentic_loop.scorer import score as real_score
from agentic_loop.loop import run as bmad_run, DEFAULT_THRESHOLD, DEFAULT_MAX_ITERS

def mps_gb():
    return torch.mps.current_allocated_memory()/1e9 if torch.backends.mps.is_available() else 0.0

def evict(model_key, framework):
    """Section 4.5.2 memory management: drop the cached adapter and free MPS memory."""
    before = mps_gb()
    key = (model_key, framework)
    if key in generator._model_cache:
        tok, mdl = generator._model_cache.pop(key)
        del tok, mdl
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    after = mps_gb()
    print(f"[memory mgmt] evicted {model_key}/{framework}:  "
          f"{before:.2f} GB -> {after:.2f} GB")
    print("  models still resident:", list(generator._model_cache.keys()) or "none")

print("MPS available:", torch.backends.mps.is_available(),
      "| threshold:", DEFAULT_THRESHOLD, "| max iters:", DEFAULT_MAX_ITERS)

MPS available: True | threshold: 0.6 | max iters: 3


## 1. Sample user stories

In [2]:

stories = [
    {
        "id": "US-101",
        "category": "Authentication",
        "complexity": "simple",
        "text": "As a registered user, I want to log in with my email and password so that I can access my dashboard.",
        "reference": {
            "cypress": '''describe('Login', () => {
  it('logs in with valid credentials', () => {
    cy.visit('/login');
    cy.get('[data-testid="email-input"]').type('user@example.com');
    cy.get('[data-testid="password-input"]').type('Secret123!');
    cy.get('[data-testid="login-button"]').click();
    cy.url().should('include', '/dashboard');
    cy.get('[data-testid="welcome-message"]').should('be.visible');
  });
});''',
            "playwright": '''import { test, expect } from '@playwright/test';

test('logs in with valid credentials', async ({ page }) => {
  await page.goto('/login');
  await page.getByLabel('Email').fill('user@example.com');
  await page.getByLabel('Password').fill('Secret123!');
  await page.getByRole('button', { name: 'Log in' }).click();
  await expect(page).toHaveURL(/dashboard/);
  await expect(page.getByTestId('welcome-message')).toBeVisible();
});''',
        },
    },
    {
        "id": "US-142",
        "category": "CRUD Operations",
        "complexity": "medium",
        "text": "As a project manager, I want to create a new task with a title and due date so that my team knows what to work on next.",
        "reference": {
            "cypress": '''describe('Task creation', () => {
  it('creates a new task with title and due date', () => {
    cy.visit('/tasks');
    cy.get('[data-testid="new-task-button"]').click();
    cy.get('[data-testid="task-title-input"]').type('Prepare release notes');
    cy.get('[data-testid="task-due-date-input"]').type('2026-08-15');
    cy.get('[data-testid="save-task-button"]').click();
    cy.get('[data-testid="task-list"]').should('contain', 'Prepare release notes');
  });
});''',
            "playwright": '''import { test, expect } from '@playwright/test';

test('creates a new task with title and due date', async ({ page }) => {
  await page.goto('/tasks');
  await page.getByRole('button', { name: 'New task' }).click();
  await page.getByLabel('Title').fill('Prepare release notes');
  await page.getByLabel('Due date').fill('2026-08-15');
  await page.getByRole('button', { name: 'Save' }).click();
  await expect(page.getByTestId('task-list')).toContainText('Prepare release notes');
});''',
        },
    },
    {
        "id": "US-207",
        "category": "Form Validation",
        "complexity": "medium",
        "text": "As a new user, I want to see an inline error if I submit the signup form with a mismatched password confirmation so that I can correct it immediately.",
        "reference": {
            "cypress": '''describe('Signup form validation', () => {
  it('shows an error when password confirmation does not match', () => {
    cy.visit('/signup');
    cy.get('[data-testid="password-input"]').type('Secret123!');
    cy.get('[data-testid="confirm-password-input"]').type('Different123!');
    cy.get('[data-testid="signup-button"]').click();
    cy.get('[data-testid="password-error"]').should('be.visible')
      .and('contain', 'Passwords do not match');
  });
});''',
            "playwright": '''import { test, expect } from '@playwright/test';

test('shows an error when password confirmation does not match', async ({ page }) => {
  await page.goto('/signup');
  await page.getByLabel('Password', { exact: true }).fill('Secret123!');
  await page.getByLabel('Confirm password').fill('Different123!');
  await page.getByRole('button', { name: 'Sign up' }).click();
  await expect(page.getByTestId('password-error')).toBeVisible();
  await expect(page.getByTestId('password-error')).toContainText('Passwords do not match');
});''',
        },
    },
    {
        "id": "US-233",
        "category": "Responsive Design",
        "complexity": "complex",
        "text": "As a mobile user, I want to open the navigation via a hamburger menu so that I can reach other pages on a small screen.",
        "reference": {
            "cypress": '''describe('Mobile navigation', () => {
  it('opens the nav menu via the hamburger button on a small viewport', () => {
    cy.viewport('iphone-x');
    cy.visit('/');
    cy.get('[data-testid="hamburger-menu-button"]').click();
    cy.get('[data-testid="mobile-nav"]').should('be.visible');
    cy.get('[data-testid="mobile-nav"]').contains('Products').click();
    cy.url().should('include', '/products');
  });
});''',
            "playwright": '''import { test, expect } from '@playwright/test';

test.use({ viewport: { width: 375, height: 812 } });

test('opens the nav menu via the hamburger button on a small viewport', async ({ page }) => {
  await page.goto('/');
  await page.getByTestId('hamburger-menu-button').click();
  await expect(page.getByTestId('mobile-nav')).toBeVisible();
  await page.getByTestId('mobile-nav').getByText('Products').click();
  await expect(page).toHaveURL(/products/);
});''',
        },
    },
]

import pandas as pd
pd.DataFrame([{"ID": s["id"], "Category": s["category"], "Complexity": s["complexity"], "Story": s["text"]} for s in stories])


,ID,Category,Complexity,Story
0,US-101,Authentication,simple,"As a registered user, I want to log in with my..."
1,US-142,CRUD Operations,medium,"As a project manager, I want to create a new t..."
2,US-207,Form Validation,medium,"As a new user, I want to see an inline error i..."
3,US-233,Responsive Design,complex,"As a mobile user, I want to open the navigatio..."


## Syntax validation method (defined once, used in every part below)

In [3]:

import subprocess, tempfile, os

def check_syntax(candidate, framework):
    ext = ".cy.js" if framework == "cypress" else ".spec.ts"
    # node --check parses JS; for a live, dependency-free check we validate the
    # .ts sample as plain JS syntax (import/await/async are valid ES module
    # syntax under --input-type=module), matching the dissertation's actual
    # approach of parsing with Node's own engine rather than a framework-specific
    # compiler.
    with tempfile.NamedTemporaryFile(suffix=".mjs", mode="w", delete=False) as f:
        f.write(candidate)
        path = f.name
    try:
        result = subprocess.run(
            ["node", "--check", path],
            capture_output=True, text=True, timeout=10,
        )
        return result.returncode == 0, result.stderr.strip()
    finally:
        os.unlink(path)

ok, err = check_syntax(stories[0]["reference"]["cypress"], "cypress")
print("Reference script valid:", ok, err or "(no errors)")

# Demonstrate the exact defect the dissertation reports catching (Section 5.2):
# an unstripped Markdown code fence corrupting the syntax check. This needs a
# script that uses a backtick template literal for a parameterised selector
# (a realistic pattern -- e.g. selecting a row by id) for the fence's stray
# backticks to actually collide with the code's own backticks the way the
# dissertation describes, rather than merely wrapping the code in an inert
# (still-parseable) string.
templated_script = '''import { test, expect } from '@playwright/test';

test(`selects the row for id ${1}`, async ({ page }) => {
  await page.goto('/rows/1');
  await expect(page.getByTestId(`row-1`)).toBeVisible();
});'''

ok_clean, err_clean = check_syntax(templated_script, "playwright")
print("\nUn-fenced templated script valid:", ok_clean, err_clean or "(no errors)")

fenced = "```typescript\n" + templated_script + "\n```"
ok_fenced, err_fenced = check_syntax(fenced, "playwright")
print("\nSame script wrapped in an un-stripped Markdown fence:")
print("Valid:", ok_fenced)
print("Error:", err_fenced[:300])


Reference script valid: True (no errors)

Un-fenced templated script valid: True (no errors)

Same script wrapped in an un-stripped Markdown fence:
Valid: False
Error: /private/var/folders/wf/70jc_wf11vn1cxvhnl5dbc2xjrw25j/T/tmpy41dxge6.mjs:4
test(`selects the row for id ${1}`, async ({ page }) => {
      ^^^^^^^

SyntaxError: Unexpected identifier 'selects'
    at checkSyntax (node:internal/main/check_syntax:72:5)

Node.js v24.3.0


## Part A — Phi-3 Mini / Cypress  (Sections 2–5)
Build → Measure → Syntax → BMAD loop for **phi3 / cypress**, on story US-101.

In [4]:
MODEL, FW = "phi3", "cypress"
s = stories[0]

# --- Section 2: BUILD (real fine-tuned adapter; first call loads the model) ---
script, latency = real_generate(MODEL, FW, s["text"], s["category"], s["complexity"], 512)
print(f"[BUILD]   {MODEL}/{FW} generated in {latency}s ({len(script)} chars)")

# --- Section 3: MEASURE (real composite scorer) ---
s_q = real_score(script, FW, exemplar=s["reference"][FW])
print(f"[MEASURE] composite={s_q.total:.3f}  "
      f"(syntax={s_q.syntax:.2f}, assertion={s_q.assertion:.2f}, rougeL={s_q.rouge_l:.2f}, complete={s_q.complete})")

# --- Section 4: SYNTAX (independent node --check) ---
ok, err = check_syntax(script, FW)
print("[SYNTAX]  node --check:", "valid" if ok else ("INVALID -> " + err[:120]))

print("\n----- generated script -----\n" + script)

⏳ Loading phi3/cypress on mps...


/Users/saif.afzal/Documents/Dissertation/agentic-test-gen/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 195/195 [00:00<00:00, 5862.88it/s]


✅ phi3/cypress ready
[BUILD]   phi3/cypress generated in 110.458s (2072 chars)
[MEASURE] composite=0.689  (syntax=0.85, assertion=1.00, rougeL=0.16, complete=True)
[SYNTAX]  node --check: valid

----- generated script -----
describe('User Login with Email and Password', () => {
  beforeEach(() => {
    cy.visit('/login');
  });

  it('should successfully log in with valid email and password and redirect to dashboard', () => {
    // Intercept the login API call
    cy.intercept('POST', '/api/auth/login').as('loginRequest');

    // Fill login form with valid credentials
    cy.get('[data-testid="email-input"]').type('user@example.com');
    cy.get('[data-testid="password-input"]').type('SecurePassword123!');

    // Click login button
    cy.get('[data-testid="login-button"]').click();

    // Wait for API response
    cy.wait('@loginRequest');

    // Verify success message appears
    cy.get('[data-testid="login-success-message"]').should('be.visible').should('contain', 'Login succes

In [5]:
# --- Section 5: the REAL BMAD loop (Build->Measure->Assess->Decide) ---
r = bmad_run(tc_id=s["id"], user_story=s["text"], framework=FW, model_key=MODEL, max_tokens=512,
             category=s["category"], complexity=s["complexity"], exemplar=s["reference"][FW])
print(f"[BMAD] {r.tc_id}/{FW}  accepted={r.accepted}  score={r.best_score:.3f}  "
      f"iters={r.iterations}  ({r.total_latency_s:.1f}s)")

[BMAD] US-101/cypress  accepted=True  score=0.689  iters=1  (218.3s)


**Memory management** — release the phi3/cypress adapter before the next combination (Section 4.5.2).

In [6]:
evict("phi3", "cypress")

[memory mgmt] evicted phi3/cypress:  7.75 GB -> 0.00 GB
  models still resident: none


## Part B — Phi-3 Mini / Playwright  (Sections 2–5)
Build → Measure → Syntax → BMAD loop for **phi3 / playwright**, on story US-101.

In [7]:
MODEL, FW = "phi3", "playwright"
s = stories[0]

# --- Section 2: BUILD (real fine-tuned adapter; first call loads the model) ---
script, latency = real_generate(MODEL, FW, s["text"], s["category"], s["complexity"], 512)
print(f"[BUILD]   {MODEL}/{FW} generated in {latency}s ({len(script)} chars)")

# --- Section 3: MEASURE (real composite scorer) ---
s_q = real_score(script, FW, exemplar=s["reference"][FW])
print(f"[MEASURE] composite={s_q.total:.3f}  "
      f"(syntax={s_q.syntax:.2f}, assertion={s_q.assertion:.2f}, rougeL={s_q.rouge_l:.2f}, complete={s_q.complete})")

# --- Section 4: SYNTAX (independent node --check) ---
ok, err = check_syntax(script, FW)
print("[SYNTAX]  node --check:", "valid" if ok else ("INVALID -> " + err[:120]))

print("\n----- generated script -----\n" + script)

⏳ Loading phi3/playwright on mps...


Loading weights: 100%|██████████| 195/195 [00:00<00:00, 5225.36it/s]


✅ phi3/playwright ready
[BUILD]   phi3/playwright generated in 221.428s (3339 chars)
[MEASURE] composite=0.745  (syntax=1.00, assertion=1.00, rougeL=0.15, complete=True)
[SYNTAX]  node --check: valid

----- generated script -----
import { test, expect } from '@playwright/test';

test.describe('User Login with Email and Password', () => {
  test.beforeEach(async ({ page }) => {
    await page.goto('/login');
  });

  test('should successfully log in with valid email and password and navigate to dashboard', async ({ page }) => {
    // Setup: intercept the login API call
    await page.route('**/api/auth/login', (route) => {
      route.continue();
    });

    // Get email input by role
    const emailInput = page.getByLabel('Email address');
    await expect(emailInput).toBeVisible();
    await emailInput.fill('user@example.com');

    // Get password input by role
    const passwordInput = page.getByLabel('Password');
    await expect(passwordInput).toBeVisible();
    await passwordIn

In [8]:
# --- Section 5: the REAL BMAD loop (Build->Measure->Assess->Decide) ---
r = bmad_run(tc_id=s["id"], user_story=s["text"], framework=FW, model_key=MODEL, max_tokens=512,
             category=s["category"], complexity=s["complexity"], exemplar=s["reference"][FW])
print(f"[BMAD] {r.tc_id}/{FW}  accepted={r.accepted}  score={r.best_score:.3f}  "
      f"iters={r.iterations}  ({r.total_latency_s:.1f}s)")

[BMAD] US-101/playwright  accepted=True  score=0.745  iters=1  (50.8s)


**Memory management** — release the phi3/playwright adapter before the next combination (Section 4.5.2).

In [ ]:
evict("phi3", "playwright")

[memory mgmt] evicted phi3/playwright:  7.75 GB -> 0.00 GB
  models still resident: none


: 